# Explicabilidade acústica de detecção de deepfake (pt-BR): zero-shot vs. adaptado

Este notebook explica, em termos acústicos, **em que o detector XLS-R se apoia** para decidir, e
contrasta o modelo **zero-shot** (`D_zs`) com uma **adaptação leve** (`D_ad`: XLS-R congelado +
regressão logística sobre embeddings em pt-BR).

Duas espinhas de explicabilidade, aplicadas a ambos os detectores:
1. **Surrogate + SHAP** — uma árvore imita o `P(spoof)` do detector a partir de 26 features MFCC.
2. **Oclusão causal espectral** — perturba bandas de frequência do áudio e mede a queda de `P(spoof)`.



## Infraestrutura reutilizada

> **Nota.** As celulas de *setup*, *pre-processamento*, *modelo*, *dados*, *inferencia* e *metricas*
> abaixo sao adaptadas (copiadas) de `deepfake_brspeech_zeroshot.ipynb`. Elas serao de-duplicadas
> durante a futura modularizacao em `scripts/`.


## 1. Setup: dependencias, imports e configuracao

Instalamos versoes fixadas para reprodutibilidade. O front-end SSL dos checkpoints anti-spoofing
publicos (colecao AntiDeepfake e W2V2-AASIST) e construido via **fairseq**, por isso ele e uma
dependencia obrigatoria.

**Atencao ao instalar o fairseq:** ele depende de `omegaconf<2.1`, cujos wheels tem metadata legada
que o `pip>=24.1` recusa (erro `ResolutionImpossible` / "invalid metadata"). Por isso a celula
abaixo primeiro rebaixa o pip para `<24.1` e so entao instala o fairseq. Ele tambem exige `numpy<2`;
se o Colab avisar, reinicie a sessao uma unica vez apos a instalacao.

In [ ]:
import os

# No container Docker (ver Dockerfile) as dependencias ja estao instaladas na imagem e a
# variavel IN_DOCKER=1 e definida, entao pulamos a instalacao. Fora dele (ex.: Colab)
# instalamos normalmente.
if os.environ.get("IN_DOCKER"):
    print("Ambiente Docker detectado: dependencias ja instaladas na imagem; pulando %pip.")
else:
    # O fairseq 0.12.2 depende de omegaconf<2.1, cujos wheels tem metadata legada que o
    # pip>=24.1 recusa ("ResolutionImpossible"). A correcao canonica no Colab e rebaixar o pip
    # para <24.1 ANTES de instalar o fairseq.
    %pip install -q "pip<24.1"

    # Com o pip compativel, instalamos o restante. numpy<2 tambem e exigido pelo fairseq.
    %pip install -q "numpy<2" "fairseq==0.12.2" "huggingface-hub>=0.31" \
        "datasets>=2.19" "transformers>=4.40" "torchaudio" "soundfile>=0.12" \
        "scikit-learn>=1.3" "webrtcvad" "pydub" "pandas" "matplotlib" "tqdm"

    # ffmpeg (usado pelo pydub para a degradacao MP3) ja vem instalado no Colab.
    # Se estiver rodando localmente, instale-o pelo gerenciador de pacotes do seu SO.
    #
    # NOTA sobre os avisos do pip: sucesso = a linha "Building wheel for fairseq ... done".
    # Depois dela, o pip pode listar CONFLITOS de dependencia do tipo:
    #     "shap/opencv/jax/cupy/tifffile ... requires numpy>=2, but you have numpy 1.26.4".
    # Esses sao apenas AVISOS: o fairseq exige numpy<2, e esses pacotes ja pre-instalados no Colab
    # preferem numpy>=2. Como este notebook NAO usa nenhum deles, os avisos sao inofensivos.
    #
    # IMPORTANTE: por causa do downgrade de numpy, use "Runtime > Restart session" UMA vez apos esta
    # celula e entao continue a partir da celula de imports (nao precisa reinstalar).

In [ ]:
from __future__ import annotations

import dataclasses
import io
import random
import sys
import warnings
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import torchaudio
import webrtcvad
from datasets import Audio, load_dataset
from huggingface_hub import PyTorchModelHubMixin
from pydub import AudioSegment
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    roc_curve,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


@dataclass(frozen=True)
class Config:
    """Parametros globais do experimento, centralizados para facilitar ajuste e reproducao.

    Attributes:
        sample_rate: Taxa de amostragem exigida pelo modelo (16 kHz e o padrao para wav2vec 2.0).
        num_samples: Tamanho fixo da janela de audio, em amostras. 64.600 ~ 4,04 s a 16 kHz,
            configuracao usada por Tak et al. (2022).
        checkpoint: ID do checkpoint anti-spoofing no Hugging Face Hub (front-end + back-end).
        dataset_id: ID do dataset em portugues no Hugging Face Hub.
        dataset_config: Nome da configuracao do dataset.
        eval_split: Split usado para avaliacao (zero-shot, sem treino).
        n_per_class: Numero de audios por classe (bonafide/spoof) no subconjunto balanceado.
        seed: Semente global de aleatoriedade.
        device: "cuda" se houver GPU disponivel, senao "cpu".
    """

    sample_rate: int = 16_000
    num_samples: int = 64_600
    checkpoint: str = "nii-yamagishilab/mms-300m-anti-deepfake"
    dataset_id: str = "AKCIT-Deepfake/BRSpeech-DF"
    dataset_config: str = "default"
    eval_split: str = "test"
    n_per_class: int = 200
    seed: int = 42
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")


CFG = Config()


def set_seed(seed: int = CFG.seed) -> None:
    """Fixa as sementes de aleatoriedade de Python, NumPy e PyTorch.

    Garante que a amostragem do dataset, a adicao de ruido e qualquer outra operacao
    estocastica sejam reproduziveis entre execucoes.

    Args:
        seed: Valor da semente a ser aplicado em todas as bibliotecas.

    Returns:
        None. O efeito e colateral (altera o estado global dos geradores).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
# Sanidade do ambiente: o fairseq exige numpy<2. Se o import de torch/torchaudio acima falhou
# com erro de ABI do numpy, reinicie a sessao (Runtime > Restart session) e rode esta celula de novo.
assert np.__version__.startswith("1."), (
    f"numpy {np.__version__} detectado; o fairseq precisa de numpy<2. "
    "Rode a celula de instalacao e reinicie a sessao."
)
print(f"numpy / torch  : {np.__version__} / {torch.__version__}")
print(f"Dispositivo    : {CFG.device}")
print(f"Checkpoint     : {CFG.checkpoint}")
print(f"Janela de audio: {CFG.num_samples} amostras (~{CFG.num_samples / CFG.sample_rate:.2f} s)")

## 2. Pre-processamento do audio bruto

O wav2vec 2.0 recebe a **forma de onda bruta** diretamente. Padronizamos todo audio em quatro
passos, todos implementados como funcoes puras (facil de testar e depurar):

1. `to_mono` - colapsa canais estereo em mono.
2. `resample_to_16k` - reamostra para 16 kHz.
3. `fix_length` - corta ou repete (tile) para exatamente 64.600 amostras.
4. `normalize_waveform` - normalizacao media/variancia (`layer_norm`, padrao dos checkpoints
   AntiDeepfake) ou, opcionalmente, normalizacao de pico.

`preprocess` encadeia os quatro passos e e a unica funcao chamada no restante do notebook.

In [ ]:
def to_mono(waveform: torch.Tensor) -> torch.Tensor:
    """Converte uma forma de onda para canal unico (mono).

    Args:
        waveform: Tensor de audio. Pode ter shape (amostras,), (canais, amostras)
            ou (1, amostras).

    Returns:
        Tensor 1-D (amostras,) contendo a media dos canais.
    """
    if waveform.ndim == 2:
        waveform = waveform.mean(dim=0)
    return waveform.reshape(-1)


def resample_to_16k(waveform: torch.Tensor, orig_sr: int, target_sr: int = CFG.sample_rate) -> torch.Tensor:
    """Reamostra a forma de onda para a taxa alvo (16 kHz por padrao).

    Args:
        waveform: Tensor 1-D de audio.
        orig_sr: Taxa de amostragem original do audio, em Hz.
        target_sr: Taxa de amostragem desejada, em Hz.

    Returns:
        Tensor 1-D reamostrado. Se `orig_sr == target_sr`, retorna o tensor inalterado.
    """
    if orig_sr == target_sr:
        return waveform
    return torchaudio.functional.resample(waveform, orig_freq=orig_sr, new_freq=target_sr)


def fix_length(waveform: torch.Tensor, num_samples: int = CFG.num_samples) -> torch.Tensor:
    """Ajusta a forma de onda para um numero exato de amostras.

    Se o audio for mais longo, mantem apenas as primeiras `num_samples` amostras (janela
    deterministica, igual a usada na avaliacao de Tak et al.). Se for mais curto, repete o
    sinal (tile) ate atingir o tamanho e entao corta o excedente.

    Args:
        waveform: Tensor 1-D de audio.
        num_samples: Comprimento fixo desejado, em amostras.

    Returns:
        Tensor 1-D com exatamente `num_samples` amostras.
    """
    length = waveform.shape[0]
    if length >= num_samples:
        return waveform[:num_samples]
    n_repeats = num_samples // length + 1
    return waveform.repeat(n_repeats)[:num_samples]


def normalize_waveform(waveform: torch.Tensor, mode: str = "layernorm") -> torch.Tensor:
    """Normaliza a forma de onda antes de alimentar o modelo.

    Args:
        waveform: Tensor 1-D de audio.
        mode: Estrategia de normalizacao:
            - "layernorm": normalizacao media/variancia (equivalente a
              `torch.nn.functional.layer_norm`), usada pelos checkpoints AntiDeepfake.
            - "peak": normalizacao de pico (divide pelo maximo valor absoluto), citada por
              Tak et al. como alternativa.

    Returns:
        Tensor 1-D normalizado.

    Raises:
        ValueError: Se `mode` nao for "layernorm" nem "peak".
    """
    if mode == "layernorm":
        return F.layer_norm(waveform, waveform.shape)
    if mode == "peak":
        peak = waveform.abs().max()
        return waveform / peak if peak > 0 else waveform
    raise ValueError(f"mode invalido: {mode!r}. Use 'layernorm' ou 'peak'.")


def preprocess(
    audio_array: np.ndarray,
    orig_sr: int,
    num_samples: int = CFG.num_samples,
    norm_mode: str = "layernorm",
) -> torch.Tensor:
    """Aplica o pipeline completo de pre-processamento a um array de audio.

    Encadeia: mono -> reamostragem 16 kHz -> janela fixa -> normalizacao.

    Args:
        audio_array: Audio bruto como array NumPy (float). 1-D ou (canais, amostras).
        orig_sr: Taxa de amostragem original, em Hz.
        num_samples: Comprimento fixo de saida, em amostras.
        norm_mode: Estrategia de normalizacao repassada a `normalize_waveform`.

    Returns:
        Tensor 1-D `float32` pronto para o modelo, com `num_samples` amostras a 16 kHz.
    """
    waveform = torch.as_tensor(audio_array, dtype=torch.float32)
    waveform = to_mono(waveform)
    waveform = resample_to_16k(waveform, orig_sr)
    waveform = fix_length(waveform, num_samples)
    waveform = normalize_waveform(waveform, mode=norm_mode)
    return waveform


# Verificacao rapida do pipeline com um sinal senoidal sintetico.
_demo = preprocess(np.sin(np.linspace(0, 100, 8000)).astype(np.float32), orig_sr=8_000)
print(f"preprocess OK -> shape={tuple(_demo.shape)} | media={_demo.mean():.3e} | std={_demo.std():.3f}")

## 3. Modelo: front-end XLS-R + back-end simples (pooling + linear)

Reproduzimos a arquitetura dos checkpoints **AntiDeepfake** (Yamagishi Lab): um front-end SSL
wav2vec 2.0 (construido via fairseq) seguido de um back-end minimalista `AdaptiveAvgPool1d + Linear`,
exatamente o "back-end de baixo esforco" descrito por Tak et al. (2022). Os pesos ja vem ajustados
para a tarefa; **nao treinamos nada**.

O modelo devolve 2 logits. Convencao do checkpoint AntiDeepfake: **classe 0 = fake (spoof)** e
**classe 1 = real (bonafide)**. O wrapper `AntiDeepfakeDetector` padroniza a saida expondo sempre a
**probabilidade de spoof**, alem de dar acesso as camadas do transformer (necessario para o GATR).

O `checkpoint` e trocavel: alem do `mms-300m-anti-deepfake` (multilingue), qualquer modelo da
mesma familia (ex.: `wav2vec-large-anti-deepfake`) usa a mesma classe. Ao final ha um bloco
**comentado** mostrando como usar o `W2V2-AASIST` (XLS-R treinado so em ASVspoof/ingles), que segue
uma narrativa zero-shot cross-lingual mais "pura", porem com back-end AASIST.

> Nota de compatibilidade: o fairseq 0.12.2 e o hydra 1.0.x nao importam em Python >= 3.11 por
> causa de defaults mutaveis em dataclasses. A funcao `install_dataclass_mutable_default_shim()`
> (no inicio da celula abaixo) instala um shim no modulo `dataclasses` que converte esses defaults
> em `field(default_factory=...)` automaticamente, cobrindo fairseq, hydra e omegaconf de uma vez.

In [ ]:
# --- Compatibilidade fairseq/hydra x Python >= 3.11 --------------------------------------------
# Em Python 3.11+, @dataclass proibe default mutavel (instancia de dataclass, dict, list, ...),
# quebrando o import do fairseq 0.12.2 E do hydra 1.0.x com:
#   "ValueError: mutable default ... is not allowed: use default_factory".
# Em vez de editar arquivo por arquivo (fairseq, hydra, omegaconf...), instalamos um shim no
# proprio modulo `dataclasses`: ao detectar um campo com default mutavel, convertemos para
# field(default_factory=...) automaticamente. Cobre todos os pacotes de uma vez. Idempotente.
def install_dataclass_mutable_default_shim() -> bool:
    """Permite dataclasses com defaults mutaveis (comportamento pre-3.11) via monkeypatch.

    Envolve `dataclasses._get_field`. Trata SOMENTE o padrao que quebra fairseq/hydra: um campo
    cujo default e uma INSTANCIA de dataclass nao-hashable (esteja "cru" no atributo ou embrulhado
    em `field(default=...)`). Nesse caso, torna a classe do default hashable, o que satisfaz a
    checagem do @dataclass e PRESERVA a instancia em `.default` - essencial porque o fairseq le
    `FairseqConfig.__dataclass_fields__[k].default` diretamente no `hydra_init`.

    Defaults mutaveis que NAO sao dataclass (dict/list) e campos ClassVar/InitVar sao deixados
    intactos, para nao interferir em bibliotecas como `datasets` (que usa ClassVar com lista).
    Necessario para importar fairseq 0.12.2 e hydra 1.0.x em Python 3.11+.

    Returns:
        True se o shim foi instalado agora; False se ja estava instalado.
    """
    if getattr(dataclasses, "_mutable_default_shim_installed", False):
        return False
    original_get_field = dataclasses._get_field

    def _get_field_with_shim(cls, a_name, a_type, *args, **kwargs):
        raw = getattr(cls, a_name, dataclasses.MISSING)
        # O default problematico do fairseq/hydra pode estar "cru" no atributo OU embrulhado em
        # field(default=...). Tratamos APENAS instancias de dataclass; dict/list/ClassVar ficam
        # intocados para nao quebrar libs como `datasets` (que usa ClassVar com default de lista).
        candidate = raw.default if isinstance(raw, dataclasses.Field) else raw
        if (
            candidate is not dataclasses.MISSING
            and not isinstance(candidate, type)
            and dataclasses.is_dataclass(candidate)
            and getattr(type(candidate), "__hash__", None) is None
        ):
            # Torna a classe hashable e preserva a instancia em `.default` (o fairseq le isso).
            type(candidate).__hash__ = object.__hash__
        return original_get_field(cls, a_name, a_type, *args, **kwargs)

    dataclasses._get_field = _get_field_with_shim
    dataclasses._mutable_default_shim_installed = True
    return True


def reset_modules(prefixes: tuple[str, ...]) -> None:
    """Remove de `sys.modules` os modulos cujos nomes comecam pelos prefixos dados.

    Um import que falhou antes deixa modulos parcialmente carregados em cache; limpa-los permite
    reimportar fairseq/hydra do zero, agora com o shim ativo.

    Args:
        prefixes: Prefixos de nomes de modulo a limpar (ex.: ("fairseq", "hydra")).

    Returns:
        None.
    """
    for name in [m for m in sys.modules if m.startswith(prefixes)]:
        del sys.modules[name]


install_dataclass_mutable_default_shim()
reset_modules(("fairseq", "hydra"))

# NOTA: o import do fairseq precisa ficar aqui (nao no bloco de imports da secao 1): so pode
# acontecer DEPOIS do shim de dataclasses + reset_modules acima, senao quebra em Python 3.11+.
from fairseq.models.wav2vec import Wav2Vec2Config, Wav2Vec2Model


class SSLModel(torch.nn.Module):
    """Front-end wav2vec 2.0 (XLS-R / large ~300M) construido via fairseq.

    A arquitetura e criada com pesos aleatorios; os pesos reais sao carregados depois, junto com
    o back-end, por `DeepfakeDetector.from_pretrained`. A config abaixo corresponde a arquitetura
    "large" de 24 camadas (embed_dim=1024), compativel com os checkpoints AntiDeepfake de ~300M
    (mms-300m e wav2vec-large).
    """

    def __init__(self) -> None:
        super().__init__()
        cfg = Wav2Vec2Config(
            quantize_targets=True,
            extractor_mode="layer_norm",
            layer_norm_first=True,
            final_dim=768,
            latent_temp=(2.0, 0.1, 0.999995),
            encoder_layerdrop=0.0,
            dropout_input=0.0,
            dropout_features=0.0,
            dropout=0.0,
            attention_dropout=0.0,
            conv_bias=True,
            encoder_layers=24,
            encoder_embed_dim=1024,
            encoder_ffn_embed_dim=4096,
            encoder_attention_heads=16,
            feature_grad_mult=1.0,
        )
        self.model = Wav2Vec2Model(cfg)

    def extract_feat(self, input_data: torch.Tensor) -> torch.Tensor:
        """Extrai as features por frame do encoder do wav2vec 2.0.

        Diferente do exemplo oficial, NAO usamos `torch.no_grad()` aqui: o GATR precisa que os
        gradientes fluam pelo transformer. Quem quiser apenas inferencia deve envolver a chamada
        em `torch.inference_mode()` externamente.

        Args:
            input_data: Tensor de forma (B, T) ou (B, T, 1) com o audio bruto normalizado.

        Returns:
            Tensor de features de forma (B, T', D), onde T' e o numero de frames e D o tamanho
            do embedding (1024 para modelos large).
        """
        if input_data.ndim == 3:
            input_data = input_data[:, :, 0]
        return self.model(input_data, mask=False, features_only=True)["x"]


class DeepfakeDetector(torch.nn.Module, PyTorchModelHubMixin):
    """Detector = front-end SSL + back-end (pooling global + camada linear de 2 classes).

    Estrutura identica a dos checkpoints AntiDeepfake, o que permite carregar os pesos via
    `from_pretrained`. Saida: logits de 2 classes (0 = fake, 1 = real).
    """

    def __init__(self) -> None:
        super().__init__()
        self.ssl_orig_output_dim = 1024
        self.num_classes = 2
        self.m_ssl = SSLModel()
        self.adap_pool1d = torch.nn.AdaptiveAvgPool1d(output_size=1)
        self.proj_fc = torch.nn.Linear(self.ssl_orig_output_dim, self.num_classes)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        """Calcula os logits de 2 classes para um lote de formas de onda.

        Args:
            wav: Tensor de audio de forma (B, T), normalizado e a 16 kHz.

        Returns:
            Tensor de logits de forma (B, 2). Indice 0 = fake, indice 1 = real.
        """
        emb = self.m_ssl.extract_feat(wav)   # (B, T', D)
        emb = emb.transpose(1, 2)            # (B, D, T')
        pooled = self.adap_pool1d(emb).squeeze(-1)  # (B, D)
        return self.proj_fc(pooled)         # (B, 2)


class AntiDeepfakeDetector:
    """Wrapper de alto nivel que padroniza o uso do detector no restante do notebook.

    Esconde detalhes do checkpoint (convencao de rotulos, device, pre-processamento) e expoe
    sempre a *probabilidade de spoof*, deixando as metricas e a XAI agnosticas ao modelo.

    Attributes:
        model: Instancia de `DeepfakeDetector` ja com os pesos carregados.
        spoof_index: Indice da classe "spoof" na saida de logits.
        device: Dispositivo em que o modelo roda.
    """

    def __init__(self, model: DeepfakeDetector, spoof_index: int = 0, device: str = CFG.device) -> None:
        self.model = model.to(device).eval()
        self.spoof_index = spoof_index
        self.device = device

    @property
    def transformer_layers(self) -> torch.nn.ModuleList:
        """Camadas de auto-atencao do encoder (usadas pelos hooks do GATR).

        Returns:
            `ModuleList` com as `TransformerSentenceEncoderLayer` do front-end fairseq.
        """
        return self.model.m_ssl.model.encoder.layers

    def logits(self, waveform: torch.Tensor) -> torch.Tensor:
        """Passa um lote de formas de onda pelo modelo mantendo o grafo de gradientes.

        Args:
            waveform: Tensor (B, T) normalizado, no mesmo device do modelo.

        Returns:
            Tensor de logits (B, 2), diferenciavel (util para o GATR).
        """
        return self.model(waveform)

    @torch.inference_mode()
    def spoof_prob_tensor(self, waveform: torch.Tensor) -> torch.Tensor:
        """Probabilidade de spoof para um lote ja pre-processado (sem gradientes).

        Args:
            waveform: Tensor (B, T) normalizado, a 16 kHz.

        Returns:
            Tensor 1-D (B,) com a probabilidade de spoof de cada amostra.
        """
        probs = F.softmax(self.model(waveform.to(self.device)), dim=1)
        return probs[:, self.spoof_index]

    def spoof_prob(self, audio_array: np.ndarray, orig_sr: int) -> float:
        """Probabilidade de spoof de um unico audio bruto (pipeline completo).

        Args:
            audio_array: Audio bruto como array NumPy.
            orig_sr: Taxa de amostragem original do audio, em Hz.

        Returns:
            Probabilidade de spoof no intervalo [0, 1].
        """
        wav = preprocess(audio_array, orig_sr).unsqueeze(0)
        return float(self.spoof_prob_tensor(wav).item())


def build_detector(checkpoint: str = CFG.checkpoint, spoof_index: int = 0) -> AntiDeepfakeDetector:
    """Carrega um checkpoint anti-spoofing e devolve o wrapper pronto para uso.

    Args:
        checkpoint: ID do modelo no Hugging Face Hub (familia AntiDeepfake).
        spoof_index: Indice da classe spoof na saida (0 para os checkpoints AntiDeepfake).

    Returns:
        Um `AntiDeepfakeDetector` em modo de avaliacao, no device configurado.
    """
    model = DeepfakeDetector.from_pretrained(checkpoint)
    return AntiDeepfakeDetector(model, spoof_index=spoof_index, device=CFG.device)


detector = build_detector()
n_params = sum(p.numel() for p in detector.model.parameters())
print(f"Modelo carregado: {CFG.checkpoint}")
print(f"Parametros      : {n_params / 1e6:.1f} M")
print(f"Camadas transformer (GATR): {len(detector.transformer_layers)}")

# -------------------------------------------------------------------------------------------
# ALTERNATIVA (zero-shot cross-lingual "puro", XLS-R treinado so em ASVspoof/ingles):
# W2V2-AASIST usa um back-end AASIST proprio e nao a classe DeepfakeDetector acima, entao exige
# baixar os arquivos de rede do repositorio. Descomente para usar (mais pesado no Colab):
#
#   !git clone https://huggingface.co/SpeechAntiSpoofingBenchmarks/W2V2-AASIST
#   # ... importar W2V2AASIST de w2v2_aasist.py e adaptar um wrapper com a mesma interface
#   #     (transformer_layers / logits / spoof_prob). A classe positiva de bonafide e o indice 1,
#   #     logo spoof_index=0 aqui tambem, apos converter o score para probabilidade.
# -------------------------------------------------------------------------------------------

## 4. Dados: BRSpeech-DF (portugues) em streaming

Carregamos o split de **teste** em modo streaming (sem baixar tudo) e montamos um subconjunto
**balanceado** (`n_per_class` audios por classe). Antes de qualquer metrica, inspecionamos e
**imprimimos o mapeamento de rotulos** do dataset para saber com certeza qual inteiro representa
"spoof" e qual representa "bonafide" - esse ponto e uma fonte classica de bugs.

Guardamos as formas de onda **brutas** (nao pre-processadas), para reutiliza-las nas degradacoes
e na analise de explicabilidade.

In [ ]:
def decode_audio(audio_field) -> tuple[np.ndarray, int]:
    """Decodifica o campo de audio do dataset sem depender do torchcodec.

    Versoes recentes do `datasets` exigem o `torchcodec` para decodificar a feature `Audio`,
    e ele nao tem build compativel com o torch desta imagem. Por isso carregamos a coluna com
    `Audio(decode=False)` (que entrega bytes/caminho crus) e decodificamos aqui com `soundfile`.

    Args:
        audio_field: Valor da coluna "audio" com `decode=False`; um dict com as chaves
            "bytes" e/ou "path".

    Returns:
        Tupla `(waveform, sample_rate)` com o audio em float32 e sua taxa de amostragem nativa.
    """
    data = audio_field.get("bytes")
    source = io.BytesIO(data) if data is not None else audio_field["path"]
    waveform, sample_rate = sf.read(source, dtype="float32", always_2d=False)
    return np.asarray(waveform, dtype=np.float32), int(sample_rate)


def resolve_spoof_label(dataset_split) -> tuple[int, dict[int, str]]:
    """Descobre qual valor inteiro do rotulo corresponde a "spoof".

    Tenta ler os nomes das classes (feature `ClassLabel`). Se um dos nomes contiver "spoof",
    "fake" ou "deepfake", esse indice e tratado como spoof. Caso nao seja possivel inferir,
    assume a convencao comum do BRSpeech-DF (0 = bonafide, 1 = spoof) e emite um aviso.

    Args:
        dataset_split: Split do dataset (streaming ou nao) com atributo `.features`.

    Returns:
        Uma tupla `(spoof_label, mapping)` onde `spoof_label` e o inteiro da classe spoof e
        `mapping` associa cada inteiro ao nome legivel da classe (ou "?" se desconhecido).
    """
    label_feature = getattr(dataset_split, "features", {}).get("label")
    names = getattr(label_feature, "names", None)
    if names:
        mapping = {i: name for i, name in enumerate(names)}
        for i, name in mapping.items():
            if any(tag in name.lower() for tag in ("spoof", "fake", "deepfake")):
                return i, mapping
    warnings.warn("Nao foi possivel inferir os nomes das classes; assumindo 1 = spoof, 0 = bonafide.")
    return 1, {0: "bonafide?", 1: "spoof?"}


def collect_balanced_samples(
    dataset_split,
    n_per_class: int,
    target_sr: int = CFG.sample_rate,
) -> tuple[list[np.ndarray], list[int], list[int]]:
    """Coleta um subconjunto balanceado de audios do split em streaming.

    Percorre o stream ate reunir `n_per_class` exemplos de cada rotulo. Os audios sao decodificados
    na taxa de amostragem nativa (sem outro pre-processamento); a conversao para 16 kHz ocorre
    depois, no pre-processamento.

    Args:
        dataset_split: Split iteravel do dataset (ja embaralhado).
        n_per_class: Quantidade desejada de exemplos por classe.
        target_sr: Taxa de amostragem para a qual o audio e convertido.

    Returns:
        Tupla `(audios, sample_rates, labels)` com listas paralelas: arrays de audio brutos,
        taxas de amostragem e rotulos inteiros originais do dataset.
    """
    audios: list[np.ndarray] = []
    sample_rates: list[int] = []
    labels: list[int] = []
    counts: dict[int, int] = {}

    for example in tqdm(dataset_split, desc="Coletando audios"):
        label = int(example["label"])
        if counts.get(label, 0) >= n_per_class:
            if len(counts) >= 2 and all(c >= n_per_class for c in counts.values()):
                break
            continue
        waveform, sample_rate = decode_audio(example["audio"])
        audios.append(waveform)
        sample_rates.append(sample_rate)
        labels.append(label)
        counts[label] = counts.get(label, 0) + 1

    print(f"Coleta concluida. Distribuicao por rotulo: {counts}")
    return audios, sample_rates, labels


raw_ds = load_dataset(CFG.dataset_id, CFG.dataset_config, streaming=True)
# decode=False entrega os bytes crus do audio; nos decodificamos com soundfile (ver decode_audio)
# para nao depender do torchcodec. A reamostragem para 16 kHz ocorre depois, no pre-processamento.
eval_stream = raw_ds[CFG.eval_split].cast_column("audio", Audio(decode=False))
eval_stream = eval_stream.shuffle(seed=CFG.seed, buffer_size=2_000)

SPOOF_LABEL, LABEL_NAMES = resolve_spoof_label(eval_stream)
print(f"Mapeamento de rotulos: {LABEL_NAMES}")
print(f"Rotulo tratado como SPOOF: {SPOOF_LABEL} ({LABEL_NAMES.get(SPOOF_LABEL)})")

eval_audios, eval_srs, eval_labels_raw = collect_balanced_samples(eval_stream, CFG.n_per_class)

# y = 1 quando spoof (classe positiva na deteccao), 0 quando bonafide.
y_true = np.array([1 if lbl == SPOOF_LABEL else 0 for lbl in eval_labels_raw], dtype=int)
print(f"Total de audios avaliados: {len(eval_audios)} | spoof={int(y_true.sum())} | bonafide={int((1 - y_true).sum())}")

## 5. Inferencia zero-shot

Passamos cada audio pelo pipeline de pre-processamento e pelo detector, coletando a
**probabilidade de spoof**. Como todos os audios tem o mesmo comprimento fixo, processamos em
lotes para acelerar. Esta funcao e reutilizada na secao de degradacoes.

In [ ]:
def score_audios(
    det: AntiDeepfakeDetector,
    audios: list[np.ndarray],
    sample_rates: list[int],
    batch_size: int = 8,
    norm_mode: str = "layernorm",
    desc: str = "Inferencia",
) -> np.ndarray:
    """Calcula a probabilidade de spoof para uma lista de audios brutos.

    Aplica `preprocess` a cada audio (garantindo comprimento fixo), agrupa em lotes e roda o
    detector sem gradientes.

    Args:
        det: Detector encapsulado por `AntiDeepfakeDetector`.
        audios: Lista de arrays de audio brutos.
        sample_rates: Lista paralela com a taxa de amostragem de cada audio.
        batch_size: Numero de audios processados por lote.
        norm_mode: Estrategia de normalizacao repassada a `preprocess`.
        desc: Texto exibido na barra de progresso.

    Returns:
        Array 1-D (N,) com a probabilidade de spoof de cada audio, na mesma ordem da entrada.
    """
    scores: list[float] = []
    for start in tqdm(range(0, len(audios), batch_size), desc=desc):
        batch_audios = audios[start : start + batch_size]
        batch_srs = sample_rates[start : start + batch_size]
        batch = torch.stack(
            [preprocess(a, sr, norm_mode=norm_mode) for a, sr in zip(batch_audios, batch_srs)]
        ).to(det.device)
        scores.extend(det.spoof_prob_tensor(batch).cpu().tolist())
    return np.asarray(scores, dtype=float)


spoof_scores = score_audios(detector, eval_audios, eval_srs, desc="Inferencia zero-shot")
print(f"Scores calculados: {spoof_scores.shape[0]}")
print(f"Score medio spoof (bonafide): {spoof_scores[y_true == 0].mean():.3f}")
print(f"Score medio spoof (spoof)   : {spoof_scores[y_true == 1].mean():.3f}")

## 6. Metricas de avaliacao

Reportamos as tres metricas usadas na literatura:

- **EER** (Equal Error Rate): limiar onde a taxa de falsos positivos iguala a de falsos negativos.
  Independe de limiar fixo; quanto menor, melhor.
- **MCC** (Matthews Correlation Coefficient): correlacao robusta para classificacao binaria; varia
  de -1 (pior) a +1 (perfeito), com 0 = chute aleatorio.
- **Acuracia**: fracao de acertos.

MCC e Acuracia dependem de um limiar de decisao. Reportamos os dois casos: o **limiar do ponto EER**
(operacional, especifico da tarefa) e o **limiar 0.5** (ingenuo), para transparencia.

In [ ]:
def compute_eer(scores: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    """Calcula o Equal Error Rate e o limiar correspondente.

    Usa a curva ROC e procura o ponto onde a taxa de falsos negativos (FNR = 1 - TPR) mais se
    aproxima da taxa de falsos positivos (FPR).

    Args:
        scores: Array (N,) com a probabilidade de spoof.
        labels: Array (N,) com os rotulos binarios (1 = spoof, 0 = bonafide).

    Returns:
        Tupla `(eer, threshold)`: o EER em fracao [0, 1] e o limiar de score que o atinge.
    """
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    return eer, float(thresholds[idx])


def evaluate(scores: np.ndarray, labels: np.ndarray, condition: str = "integro") -> dict[str, float]:
    """Consolida EER, MCC e Acuracia (em dois limiares) para um conjunto de scores.

    Args:
        scores: Array (N,) com a probabilidade de spoof.
        labels: Array (N,) com os rotulos binarios (1 = spoof, 0 = bonafide).
        condition: Rotulo textual da condicao avaliada (ex.: "integro", "mp3"), usado no relatorio.

    Returns:
        Dicionario com as chaves: `condition`, `eer` (%), `eer_threshold`, `mcc_eer`, `acc_eer`,
        `mcc_0.5`, `acc_0.5`.
    """
    eer, thr = compute_eer(scores, labels)
    pred_eer = (scores >= thr).astype(int)
    pred_half = (scores >= 0.5).astype(int)
    return {
        "condition": condition,
        "eer": eer * 100.0,
        "eer_threshold": thr,
        "mcc_eer": matthews_corrcoef(labels, pred_eer),
        "acc_eer": accuracy_score(labels, pred_eer),
        "mcc_0.5": matthews_corrcoef(labels, pred_half),
        "acc_0.5": accuracy_score(labels, pred_half),
    }


def plot_confusion(scores: np.ndarray, labels: np.ndarray, threshold: float, title: str) -> None:
    """Plota a matriz de confusao para um dado limiar de decisao.

    Args:
        scores: Array (N,) com a probabilidade de spoof.
        labels: Array (N,) com os rotulos binarios (1 = spoof, 0 = bonafide).
        threshold: Limiar de score acima do qual o audio e classificado como spoof.
        title: Titulo do grafico.

    Returns:
        None. Exibe o grafico via matplotlib.
    """
    preds = (scores >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["Bonafide", "Spoof"])
    ax.set_yticks([0, 1], ["Bonafide", "Spoof"])
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


clean_metrics = evaluate(spoof_scores, y_true, condition="integro")
print("=== Metricas zero-shot (audio integro) ===")
print(f"  EER            : {clean_metrics['eer']:.2f}%")
print(f"  MCC (limiar EER): {clean_metrics['mcc_eer']:+.3f}   | Acuracia: {clean_metrics['acc_eer'] * 100:.2f}%")
print(f"  MCC (limiar 0.5): {clean_metrics['mcc_0.5']:+.3f}   | Acuracia: {clean_metrics['acc_0.5'] * 100:.2f}%")

plot_confusion(spoof_scores, y_true, clean_metrics["eer_threshold"], "Matriz de confusao (limiar EER)")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Paleta segura para daltônicos (ordem estável entre figuras).
CB_PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]

def set_plot_style() -> None:
    """Define um estilo de figura consistente (padrao de paper ACM sigconf).

    Centraliza rcParams para reprodutibilidade e legibilidade em coluna de artigo.
    Chamar uma vez, no inicio; todas as figuras herdam o estilo.
    """
    mpl.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "grid.linewidth": 0.5,
        "axes.prop_cycle": mpl.cycler(color=CB_PALETTE),
    })

# Larguras de coluna do template ACM sigconf (polegadas).
COL_WIDTH_SINGLE = 3.33
COL_WIDTH_DOUBLE = 7.0

FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name: str) -> None:
    """Salva a figura em PDF vetorial (para o LaTeX) e PNG 300 DPI (preview)."""
    fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"))
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"))

set_plot_style()


In [ ]:
@torch.inference_mode()
def extract_embeddings(det: AntiDeepfakeDetector, audios: list[np.ndarray],
                       srs: list[int], batch_size: int = 8) -> np.ndarray:
    """Extrai o embedding pooled (1024-D) do XLS-R para cada audio.

    Reproduz exatamente a representacao que alimenta a cabeca linear do detector
    (AdaptiveAvgPool1d == media sobre os frames), garantindo comparabilidade D_zs vs D_ad.

    Args:
        det: detector com o front-end XLS-R congelado.
        audios: lista de arrays de audio bruto.
        srs: taxas de amostragem nativas, paralelas a `audios`.
        batch_size: tamanho do lote na GPU.

    Returns:
        Array (N, 1024) de embeddings em float32.
    """
    embs: list[np.ndarray] = []
    for i in range(0, len(audios), batch_size):
        wavs = [preprocess(a, sr) for a, sr in zip(audios[i:i+batch_size], srs[i:i+batch_size])]
        batch = torch.stack(wavs).to(det.device)                 # (B, T)
        frames = det.model.m_ssl.extract_feat(batch)             # (B, T', 1024)
        pooled = frames.mean(dim=1)                              # (B, 1024)
        embs.append(pooled.cpu().numpy().astype(np.float32))
    return np.concatenate(embs, axis=0)


In [ ]:
_demo_emb = extract_embeddings(detector, [np.sin(np.linspace(0, 200, 32000)).astype(np.float32)], [16000])
assert _demo_emb.shape == (1, 1024), _demo_emb.shape
print(f"extract_embeddings OK -> shape={_demo_emb.shape}, dtype={_demo_emb.dtype}")


## 1. Coleta de dados

Montamos **dois** subconjuntos balanceados e mutuamente exclusivos do BRSpeech-DF:

- **`train`** — usado *exclusivamente* para treinar a cabeca de regressao logistica do detector
  adaptado (`D_ad`). Nunca entra em nenhuma analise.
- **`test`** (balanceado) — e o **conjunto unico de analise**: todas as metricas, o surrogate +
  SHAP e a oclusao causal espectral rodam apenas sobre ele.

Como `train` e `test` vem de splits distintos do dataset, **nao ha vazamento** entre o que treina a
cabeca de `D_ad` e o que a avalia/explica.

In [ ]:
N_TRAIN_PER_CLASS = 1500   # ajustavel; treina a cabeca de D_ad
N_TEST_PER_CLASS = 1500    # ajustavel; conjunto de analise (SHAP, oclusao, testes)

raw_ds = load_dataset(CFG.dataset_id, CFG.dataset_config, streaming=True)
train_stream = raw_ds["train"].cast_column("audio", Audio(decode=False)).shuffle(seed=CFG.seed, buffer_size=2_000)
test_stream = raw_ds[CFG.eval_split].cast_column("audio", Audio(decode=False)).shuffle(seed=CFG.seed, buffer_size=2_000)

SPOOF_LABEL, LABEL_NAMES = resolve_spoof_label(test_stream)
print(f"Rotulo SPOOF={SPOOF_LABEL} ({LABEL_NAMES})")

train_audios, train_srs, train_lbls = collect_balanced_samples(train_stream, N_TRAIN_PER_CLASS)
test_audios, test_srs, test_lbls = collect_balanced_samples(test_stream, N_TEST_PER_CLASS)

y_train = np.array([1 if l == SPOOF_LABEL else 0 for l in train_lbls], dtype=int)
y_test = np.array([1 if l == SPOOF_LABEL else 0 for l in test_lbls], dtype=int)

In [ ]:
assert set(np.unique(y_train)) == {0, 1} and set(np.unique(y_test)) == {0, 1}
print(f"train: {len(y_train)} (spoof={y_train.sum()}) | test: {len(y_test)} (spoof={y_test.sum()})")

## 2. Adaptacao leve (D_ad) e pre-check de EER

Mantemos o front-end **XLS-R congelado** (nenhum peso do transformer e atualizado) e treinamos
apenas uma **cabeca leve** de classificacao: uma **regressao logistica** (com padronizacao previa)
sobre os embeddings pooled de 1024-D extraidos do split `train`. Esse e o detector adaptado
`D_ad`, em contraste com o zero-shot `D_zs` (a cabeca linear original do checkpoint).

O ponto-chave e: **comparar a explicabilidade de `D_zs` vs. `D_ad` so faz sentido se `D_ad` de fato
melhorar (reduzir) o EER de `D_zs`**. Se a adaptacao nao superar o zero-shot, o contraste
"antes/depois" perde forca (ver spec §3). Por isso, apos treinar `D_ad`, fazemos um **pre-check de
EER** comparando os dois detectores no mesmo conjunto de teste antes de seguir para a XAI.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train_emb = extract_embeddings(detector, train_audios, train_srs)
X_test_emb = extract_embeddings(detector, test_audios, test_srs)

# Cabeca adaptada: padroniza + regressao logistica (P(spoof) calibrado, classe positiva = spoof=1).
ad_head = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, C=1.0, random_state=CFG.seed))
ad_head.fit(X_train_emb, y_train)

def p_spoof_ad(emb: np.ndarray) -> np.ndarray:
    """P(spoof) do detector adaptado para uma matriz de embeddings (N, 1024)."""
    return ad_head.predict_proba(emb)[:, 1]

In [ ]:
# D_zs: probabilidade de spoof do detector zero-shot (reusa spoof_prob por audio).
scores_zs = np.array([detector.spoof_prob(a, sr) for a, sr in zip(test_audios, test_srs)])
scores_ad = p_spoof_ad(X_test_emb)

eer_zs, thr_zs = compute_eer(scores_zs, y_test)
eer_ad, thr_ad = compute_eer(scores_ad, y_test)
print(f"EER  D_zs={eer_zs:.4f} (thr={thr_zs:.3f}) | D_ad={eer_ad:.4f} (thr={thr_ad:.3f})")

In [ ]:
assert 0.0 <= eer_zs <= 1.0 and 0.0 <= eer_ad <= 1.0
if eer_ad >= eer_zs:
    print("AVISO: D_ad NAO superou D_zs — o contraste 'antes/depois' perde forca (ver spec §3).")
else:
    print(f"OK: adaptacao reduziu o EER em {(eer_zs - eer_ad):.4f}.")